# 01 -- Metodologia de las heuristicas H0 -> H1 -> H2 -> H3

Documento reproducible de la definicion CERRADA de las 4 politicas (auditoria +
5 casos controlados, ver conversacion de diseno). Cada regla que se describe
abajo se **demuestra en el mismo notebook** llamando a las funciones reales
sobre escenarios sinteticos minimos -- no son solo enunciados, son
verificaciones ejecutables.

| | H0 | H1 | H2 | H3 |
|---|---|---|---|---|
| Regla de decision por barco | local estricto > remoto | igual que H0 | local estricto > remoto, costo dentro de cada nivel | costo global, sin particion |
| Coordinacion entre barcos libres | orden de lista | por urgencia | por urgencia | por urgencia (costo) |
| Reserva persistente entre pasos | no | si | si | si |
| Funcion de costo | ninguna | ninguna | `C = T_pickup - W_max` (solo remoto) | `C = T_pickup - W_max` (local y remoto) |
| Forecasting / demanda futura | no | no | no | no |

Archivos: `politica_base/src/politica_base.py` (H0, sin tocar),
`heuristicas/comun/src/reservas.py` (reserva compartida),
`heuristicas/h1_reserva/src/politica_h1.py`,
`heuristicas/h2_costo_local/src/politica_h2.py`,
`heuristicas/h3_costo_global/src/politica_h3.py`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import yaml

BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE / "simulador" / "src"))
sys.path.insert(0, str(BASE / "politica_base" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "comun" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h1_reserva" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h2_costo_local" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h3_costo_global" / "src"))

from estado import Barco, EstadoSimulacion, colas_vacias
from unidades import Unidad
from politica_base import asignar_flota, politica_base
from politica_h1 import asignar_flota_h1
from politica_h2 import asignar_flota_h2
from politica_h3 import asignar_flota_h3

cfg_bb = yaml.safe_load(open(BASE / "bergen-boats" / "config" / "instance.yaml", encoding="utf-8"))
nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BASE / "bergen-boats" / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)
CAP = cfg_bb["flota"]["capacidad_pasajeros"]

print("Nodos:", nodos)
print("Capacidad por barco:", CAP)
print("Matriz de tiempos de viaje (min):")
matriz_tiempos.round(2)

Nodos: ['kleppesto', 'laksevag', 'bryggen', 'sandviken']
Capacidad por barco: 30
Matriz de tiempos de viaje (min):


,kleppesto,laksevag,bryggen,sandviken
kleppesto,0.00,5.33,11.50,9.56
laksevag,5.33,0.00,8.61,8.61
bryggen,11.50,8.61,0.00,6.83
sandviken,9.56,8.61,6.83,0.00


## H0 -- politica base (sin cambios)

1. **Demanda local** (sale del nodo donde el barco esta libre): si existe, elegir
   la de MAYOR espera del mas antiguo -- prioridad estricta sobre cualquier
   demanda remota, sin importar la magnitud.
2. **Demanda remota** (si no hay nada local): mayor espera del mas antiguo;
   empate se rompe por menor tiempo de viaje.
3. **Sin demanda**: esperar (`None`).
4. `asignar_flota` coordina varios barcos libres del MISMO paso, descontando
   localmente lo que cada uno se llevaria -- pero en el orden fijo de la
   lista, y (auditoria, Caso 1) **con un bug real**: para decisiones de
   reposicionamiento descuenta el par `(nodo_actual, decision)` en vez del
   par real de la cola remota, asi que dos barcos libres a la vez SI pueden
   terminar reposicionandose al mismo nodo por la misma razon. Se conserva
   tal cual, sin corregir -- es la referencia historica.

In [2]:
# Caso 1 -- el bug de H0 queda demostrado en vivo, no solo descrito
t = 100.0
colas = colas_vacias(nodos)
colas[("laksevag", "kleppesto")] = [Unidad(id=f"g_{i}", origen="laksevag", destino="kleppesto",
                                            minuto_llegada=t - 12.0, tamano=1) for i in range(5)]
b1 = Barco(id="B1", nodo_origen="bryggen", nodo_destino="bryggen", min_para_llegar=0.0)
b2 = Barco(id="B2", nodo_origen="sandviken", nodo_destino="sandviken", min_para_llegar=0.0)
estado = EstadoSimulacion(t_actual_min=t, dia_semana=0, barcos=[b1, b2], colas=colas)

print("H0:", asignar_flota([b1, b2], estado, matriz_tiempos, CAP))
print("-> los DOS barcos van a 'laksevag' aunque 5 personas las cubre uno solo: bug confirmado.")

H0: {'B1': 'laksevag', 'B2': 'laksevag'}
-> los DOS barcos van a 'laksevag' aunque 5 personas las cubre uno solo: bug confirmado.


## H1 = H0 + reserva persistente

Reglas de decision **identicas a H0** (ver `_mejor_local`/`_mejor_remoto` en
`politica_h1.py`, reimplementadas linea a linea contra el original).
La UNICA extension: cuando un barco se compromete con una cola remota
(reposicionamiento vacio), se crea

```
reservas[barco_id] = (origen, destino)
```

en un diccionario que **mantiene el bucle de control** (no la politica) de un
paso al siguiente -- no se recalcula. Se libera cuando el barco llega y
vuelve a estar libre (unico evento posible: el motor nunca redirige un barco
a medio camino). Mientras la reserva esta activa, descuenta la capacidad del
barco (no la cola entera) de lo que ven los demas barcos libres.

In [3]:
# Caso 1 con H1: la coordinacion evita la duplicacion
d1, r1 = asignar_flota_h1([b1, b2], estado, matriz_tiempos, CAP, {})
print("H1:", d1, " reservas creadas:", r1)
print("-> solo UN barco va, el otro espera.")

H1: {'B1': 'laksevag', 'B2': None}  reservas creadas: {'B1': ('laksevag', 'kleppesto')}
-> solo UN barco va, el otro espera.


In [4]:
# Caso 2 -- persistencia real entre pasos: DOS llamadas separadas, sin recalcular nada
t1 = 100.0
colas_p1 = colas_vacias(nodos)
colas_p1[("laksevag", "kleppesto")] = [Unidad(id=f"orig_{i}", origen="laksevag", destino="kleppesto",
                                               minuto_llegada=t1 - 2.0, tamano=1) for i in range(3)]
b1_p1 = Barco(id="B1", nodo_origen="bryggen", nodo_destino="bryggen", min_para_llegar=0.0)
estado_p1 = EstadoSimulacion(t_actual_min=t1, dia_semana=0, barcos=[b1_p1], colas=colas_p1)
dec_p1, reservas = asignar_flota_h1([b1_p1], estado_p1, matriz_tiempos, CAP, {})
print("Paso 1 -- B1 decide:", dec_p1, " reservas:", reservas)

t2 = t1 + matriz_tiempos.loc["bryggen", "laksevag"] / 2  # B1 a mitad de camino
colas_p2 = colas_vacias(nodos)
colas_p2[("laksevag", "kleppesto")] = (
    [Unidad(id=f"orig_{i}", origen="laksevag", destino="kleppesto", minuto_llegada=t1 - 2.0, tamano=1) for i in range(3)]
    + [Unidad(id=f"nuevo_{i}", origen="laksevag", destino="kleppesto", minuto_llegada=t2 - 1.0, tamano=1) for i in range(4)]
)
b1_transito = Barco(id="B1", nodo_origen="bryggen", nodo_destino="laksevag",
                     min_para_llegar=matriz_tiempos.loc["bryggen", "laksevag"] / 2, a_bordo=[])
b2_libre = Barco(id="B2", nodo_origen="sandviken", nodo_destino="sandviken", min_para_llegar=0.0)
estado_p2 = EstadoSimulacion(t_actual_min=t2, dia_semana=0, barcos=[b1_transito, b2_libre], colas=colas_p2)

dec_p2, reservas_p2 = asignar_flota_h1([b2_libre], estado_p2, matriz_tiempos, CAP, reservas)
print("Paso 2 -- reservas ENTRANTES (identicas a las del paso 1):", reservas)
print("Paso 2 -- B2 decide:", dec_p2, " (llegaron 4 personas mas a la misma cola, sigue cubierta)")
assert dec_p2["B2"] is None, "la reserva deberia impedir que B2 duplique el viaje"
print("OK -- persistencia confirmada.")

Paso 1 -- B1 decide: {'B1': 'laksevag'}  reservas: {'B1': ('laksevag', 'kleppesto')}
Paso 2 -- reservas ENTRANTES (identicas a las del paso 1): {'B1': ('laksevag', 'kleppesto')}
Paso 2 -- B2 decide: {'B2': None}  (llegaron 4 personas mas a la misma cola, sigue cubierta)
OK -- persistencia confirmada.


## H2 = H1 + costo, manteniendo la particion local/remoto (Alternativa B)

$$C(b,q) = T_{pickup}(b,q) - W_{max}(q)$$

```
si hay demanda local  -> comparar SOLO colas locales con C
si no hay demanda local -> comparar SOLO colas remotas con C
si no hay nada         -> esperar
```

Una cola remota **nunca** compite con una local. Para candidatos locales
`T_pickup=0`, asi que `C=-W_max`: minimizar C equivale a maximizar la espera
-- exactamente la Regla 1 de H0, sin nada nuevo que decidir ahi. El costo
solo aporta algo distinto de H0 en la rama remota: en vez de "mayor espera,
empate por menor viaje" (regla lexicografica), hace un trade-off continuo
entre viaje y espera.

In [5]:
# Caso 4 -- local barata vs. remota urgente: H2 NUNCA cruza, sin importar la magnitud
def escenario(espera_local, espera_remota, t4=200.0):
    colas4 = colas_vacias(nodos)
    colas4[("bryggen", "sandviken")] = [Unidad(id=f"loc_{i}", origen="bryggen", destino="sandviken",
                                                minuto_llegada=t4 - espera_local, tamano=1) for i in range(5)]
    colas4[("kleppesto", "laksevag")] = [Unidad(id=f"rem_{i}", origen="kleppesto", destino="laksevag",
                                                 minuto_llegada=t4 - espera_remota, tamano=1) for i in range(5)]
    b1 = Barco(id="B1", nodo_origen="bryggen", nodo_destino="bryggen", min_para_llegar=0.0)
    return b1, EstadoSimulacion(t_actual_min=t4, dia_semana=0, barcos=[b1], colas=colas4)

for espera_remota in [10, 30, 60, 200]:
    b1_c4, estado4 = escenario(espera_local=3, espera_remota=espera_remota)
    d = asignar_flota_h2([b1_c4], estado4, matriz_tiempos, CAP, {})[0]
    print(f"espera_remota={espera_remota:>3} min -> H2 decide: {d['B1']}  (local='sandviken')")
print("-> H2 nunca cruza a 'kleppesto', ni con 200 min de espera remota.")

espera_remota= 10 min -> H2 decide: sandviken  (local='sandviken')
espera_remota= 30 min -> H2 decide: sandviken  (local='sandviken')
espera_remota= 60 min -> H2 decide: sandviken  (local='sandviken')
espera_remota=200 min -> H2 decide: sandviken  (local='sandviken')
-> H2 nunca cruza a 'kleppesto', ni con 200 min de espera remota.


## H3 = H1 + costo GLOBAL (Alternativa A) -- sin particion

Misma formula, pero comparando TODAS las colas (locales y remotas) a la vez:

$$\text{candidatos} = \text{colas locales} \cup \text{colas remotas} \;\to\; \arg\min_q C(b,q)$$

Una demanda remota SI puede ganarle a una local. El umbral exacto (derivado
en la auditoria, Caso 4): lo remoto gana cuando

$$W_{max}^{remoto} > T_{pickup}^{remoto} + W_{max}^{local}$$

Con pickup Bryggen->Kleppesto = 11.5 min y espera local = 3 min, el umbral es
**14.5 min** -- se verifica abajo que el cruce ocurre justo ahi.

In [6]:
for espera_remota in [10, 14, 15, 30, 200]:
    b1_c4, estado4 = escenario(espera_local=3, espera_remota=espera_remota)
    d = asignar_flota_h3([b1_c4], estado4, matriz_tiempos, CAP, {})[0]
    print(f"espera_remota={espera_remota:>3} min -> H3 decide: {d['B1']}")

pickup = float(matriz_tiempos.loc["bryggen", "kleppesto"])
print(f"\nUmbral teorico = pickup({pickup:.1f}) + espera_local(3) = {pickup+3:.1f} min")
print("-> H3 cruza a 'kleppesto' justo entre 14 y 15 min, coincidiendo con el umbral.")

espera_remota= 10 min -> H3 decide: sandviken
espera_remota= 14 min -> H3 decide: sandviken
espera_remota= 15 min -> H3 decide: kleppesto
espera_remota= 30 min -> H3 decide: kleppesto
espera_remota=200 min -> H3 decide: kleppesto

Umbral teorico = pickup(11.5) + espera_local(3) = 14.5 min
-> H3 cruza a 'kleppesto' justo entre 14 y 15 min, coincidiendo con el umbral.


## Resumen

- **H0**: referencia historica, con un bug de coordinacion remota conocido y sin corregir (Caso 1).
- **H1**: agrega SOLO coordinacion (orden por urgencia + reserva persistente). Corrige el bug de H0, pero no cambia ninguna regla de decision.
- **H2**: agrega costo, preservando la particion local/remoto de H0 -- el costo solo refina la eleccion remota.
- **H3**: agrega costo global, sin particion -- permite que lo remoto compita con lo local, con un umbral cuantificable.

Ver `02_experimentos_heuristicas.ipynb` para la comparacion bajo una flota fija,
y `03_comparacion_flota.ipynb` para el barrido de tamano de flota.